This code work with the output from waste MM detection (from NC)  
The mm detection was done in Ucloud with Youran's account.  
 - Nature RS for Waste: https://www.nature.com/articles/s41467-023-37136-1
 - Github: https://github.com/DongshuoYin/garbage_dump_detection  

code include:  
Pre detction: predict from UNdata challenge
Predict: /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/RS_Waste/MMDetect_out/predict.py  
Post mmdetection:
1. Generate bounding box for outputs:  
    result.csv + Visual_out -> visualized_tiles
   path: '/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/RS_Waste'
2. Convert JPEG + CSV Back to GeoTIFF

In [1]:
import os
import cv2
import csv
import rasterio
import pandas as pd
from tqdm import tqdm
from rasterio.transform import from_origin

In [2]:
Ori_image = "/Users/wenlanzhang/Downloads/PhD_UCL/Data/RS/Maxar/20231130/" 
MMD = '/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/RS_Waste/MMDetect_out/'

Base = '/Users/wenlanzhang/Downloads/PhD_UCL/Data/RS/Maxar/'

pre = '20231130/'
post = '20240511/'

# Image: Generate bounding box for outputs

In [3]:
# Define paths
MMD_csv = MMD + "MMD_result.csv"  # MMD outut CSV file 
image_folder =  Ori_image + 'Visual_out/' # Visual cliped all images' location
vis_image_folder = MMD + "visualized_tiles"  # Output folder for MMD detected images

In [4]:
# Ensure the output directory exists
os.makedirs(vis_image_folder, exist_ok=True)

# Load detection results
df = pd.read_csv(MMD_csv, delimiter=",")  # Use "\t" if the CSV is tab-separated, else use ","
df = df.drop_duplicates()


## For only Domestic 

In [5]:
# Define color mapping (only for "domestic garbage")
DOMESTIC_GARBAGE_COLOR = (0, 0, 255)  # Red

# Filter only "domestic garbage"
df = df[df["Class"] == "domestic garbage"]

# Group the data by image name
grouped = df.groupby(lambda index: f"tile_{int(df.at[index, 'tile_row'])}_{int(df.at[index, 'tile_col'])}.jpg")

# Iterate over each group (image) with tqdm progress bar
for image_name, group in tqdm(grouped, total=len(grouped), desc="Processing images"):
    try:
        image_path = os.path.join(image_folder, image_name)

        # Check if the image exists
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image not found: {image_path}")

        # Load image
        img = cv2.imread(image_path)
        if img is None:
            raise ValueError(f"Error loading image: {image_path}")

        # Iterate over each detection in the group
        for _, row in group.iterrows():
            x_min, y_min, x_max, y_max = int(row["x_min"]), int(row["y_min"]), int(row["x_max"]), int(row["y_max"])
            confidence = row["confidence_score"]

            # Draw bounding box in red (only for "domestic garbage")
            cv2.rectangle(img, (x_min, y_min), (x_max, y_max), DOMESTIC_GARBAGE_COLOR, 2)

            # Add label text with confidence score
            label = f"Domestic Garbage ({confidence:.2f})"
            cv2.putText(img, label, (x_min, y_min - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, DOMESTIC_GARBAGE_COLOR, 2)

        # Save the new image with bounding boxes
        output_image_path = os.path.join(vis_image_folder, image_name)
        cv2.imwrite(output_image_path, img)
    except Exception as e:
        print(f"Error: {e}")

print("✅ Domestic Garbage Bounding Box Visualization Complete!")


Processing images: 100%|█████████████████████| 441/441 [00:03<00:00, 120.33it/s]

✅ Domestic Garbage Bounding Box Visualization Complete!


## For All classes

In [5]:
# Define color mapping for different classes
CLASS_COLORS = {
    "domestic garbage": (0, 0, 255),  # Red
    "construction waste": (206, 206, 206),
    'agriculture forestry': (45, 112, 18),
    'disposed garbage': (255, 0, 0)  # Blue
}

# Group the data by image name
grouped = df.groupby(lambda index: f"tile_{int(df.at[index, 'tile_row'])}_{int(df.at[index, 'tile_col'])}.jpg")

# Iterate over each group (image) with tqdm progress bar
for image_name, group in tqdm(grouped, total=len(grouped), desc="Processing images"):
    try:
        image_path = os.path.join(image_folder, image_name)

        # Check if the image exists
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image not found: {image_path}")

        # Load image
        img = cv2.imread(image_path)
        if img is None:
            raise ValueError(f"Error loading image: {image_path}")

        # Iterate over each detection in the group
        for _, row in group.iterrows():
            x_min, y_min, x_max, y_max = int(row["x_min"]), int(row["y_min"]), int(row["x_max"]), int(row["y_max"])
            confidence = row["confidence_score"]
            detected_class = row["Class"]

            # Choose color based on class
            color = CLASS_COLORS.get(detected_class, (0, 255, 0))  # Default Green

            # Draw bounding box
            cv2.rectangle(img, (x_min, y_min), (x_max, y_max), color, 2)

            # Add label text with confidence score
            label = f"{detected_class} ({confidence:.2f})"
            cv2.putText(img, label, (x_min, y_min - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # Save the new image with bounding boxes
        output_image_path = os.path.join(vis_image_folder, image_name)
        cv2.imwrite(output_image_path, img)
    except Exception as e:
        print(f"Error: {e}")

print("✅ Bounding box visualization complete!")


Processing images: 100%|█████████████████████| 576/576 [00:04<00:00, 119.17it/s]

✅ Bounding box visualization complete!


In [12]:
df[(df['tile_row'] == 16) & (df['tile_col'] == 38)]

,tile_row,tile_col,x_min,y_min,x_max,y_max,confidence_score,Class
36,16,38,91.25838,567.50366,128.29689,601.13390,0.353243,domestic garbage
37,16,38,377.46164,452.82565,469.49160,516.85040,0.336198,domestic garbage
38,16,38,328.09630,446.23680,488.52234,658.42260,0.802280,construction waste
39,16,38,354.61404,445.78873,482.51352,569.47375,0.568411,construction waste


# Image: Convert identified JPEG + CSV Back to GeoTIFF

In [6]:
def convert_jpeg_to_geotiff(Meta_csv, image_folder, output_folder):
    # Create the output folder if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Read the metadata from the CSV file
    metadata = {}
    with open(Meta_csv, 'r') as csvfile:
        reader = csv.DictReader(csvfile, delimiter=',')
        for row in reader:
            metadata[row['filename']] = {
                'left': float(row['left']),
                'top': float(row['top']),
                'pixel_size_x': float(row['pixel_size_x']),
                'pixel_size_y': float(row['pixel_size_y']),
                'crs': row['crs']
            }

    error_list = []
    # Iterate through all files in the image folder with tqdm progress bar
    for filename in tqdm(os.listdir(image_folder), desc="Converting images"):
        if filename.endswith('.jpg'):
            image_file = os.path.join(image_folder, filename)
            if filename in metadata:
                try:
                    # Read the JPEG image
                    with rasterio.open(image_file) as src:
                        img = src.read()
                        height, width = src.height, src.width

                    # Get the geographical information
                    left = metadata[filename]['left']
                    top = metadata[filename]['top']
                    pixel_size_x = metadata[filename]['pixel_size_x']
                    pixel_size_y = metadata[filename]['pixel_size_y']
                    crs = metadata[filename]['crs']

                    # Create the transformation matrix
                    transform = from_origin(left, top, pixel_size_x, pixel_size_y)

                    # Write the GeoTIFF file
                    output_file = os.path.join(output_folder, filename.replace('.jpg', '.tif'))
                    with rasterio.open(
                        output_file,
                        'w',
                        driver='GTiff',
                        height=height,
                        width=width,
                        count=img.shape[0],
                        dtype=img.dtype,
                        crs=crs,
                        transform=transform
                    ) as dst:
                        dst.write(img)
                except Exception as e:
                    error_list.append(f"Error converting {image_file}: {str(e)}")
            else:
                error_list.append(f"Metadata not found for {image_file}")

    if error_list:
        print("Errors occurred during conversion:")
        for error in error_list:
            print(error)
    else:
        print("All images were converted successfully.")

    return error_list


In [7]:
# Meta_csv = 'your_metadata.csv'
# image_folder = 'your_image_folder'
# output_folder = 'output'

vis_image_folder = MMD + 'visualized_tiles/'
Re_tif_folder = MMD + 'Visual_re_all'
# Re_tif_folder = MMD + 'Visual_re_domestic'
Meta_csv = Ori_image + 'Visual_meta.csv'
Meta_df = pd.read_csv(Meta_csv)
# Meta_df

errors = convert_jpeg_to_geotiff(Meta_csv, vis_image_folder, Re_tif_folder)

Converting images:   0%|                                | 0/576 [00:00<?, ?it/s]/opt/miniconda3/envs/Try/lib/python3.10/site-packages/rasterio/__init__.py:304: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
Converting images: 100%|██████████████████████| 576/576 [00:08<00:00, 66.69it/s]

All images were converted successfully.


# .csv: Get detected vector
尝试整个变成vector - 失败了因为太大，kernal老死
（各种方法都失败了）


还没有match to Meta  
差别：
 - MMD + MMD_result （detection）: 每行是一个box  
 - Ori_image + Visual_meta （meta）: 每行是一个image,可能有多个box  

## Match to detection out csv 

In [7]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon

# Load the detection CSV
detection_df = pd.read_csv(MMD + "MMD_result.csv")
detection_df = detection_df.drop_duplicates()  # Removes exact duplicate rows

# Load the metadata CSV
meta_df = pd.read_csv(Base + pre + "Visual_meta.csv")

# Ensure correct column names (strip spaces if needed)
detection_df.columns = detection_df.columns.str.strip()
meta_df.columns = meta_df.columns.str.strip()

# Generate `tile_id` in detection data to match metadata
detection_df["tile_id"] = detection_df["tile_row"].astype(str) + "_" + detection_df["tile_col"].astype(str)

# Merge detection data with metadata on `tile_id`
merged_df = detection_df.merge(meta_df, on="tile_id", how="left")
# merged_df

# Check if merge worked
if merged_df["left"].isna().sum() > 0:
    print("⚠️ Warning: Some detections didn't match any tile metadata!")

# Compute global geolocation of bounding box coordinates (considering tile’s geolocation)
merged_df["global_x_min"] = merged_df["left"] + (merged_df["x_min"] * merged_df["pixel_size_x"])
merged_df["global_y_min"] = merged_df["top"] - (merged_df["y_min"] * merged_df["pixel_size_y"])
merged_df["global_x_max"] = merged_df["left"] + (merged_df["x_max"] * merged_df["pixel_size_x"])
merged_df["global_y_max"] = merged_df["top"] - (merged_df["y_max"] * merged_df["pixel_size_y"])

# Debug: Print first few transformed coordinates
# print(merged_df[["tile_id", "global_x_min", "global_y_min", "global_x_max", "global_y_max"]].head())
merged_df

,tile_row,tile_col,x_min,y_min,x_max,y_max,confidence_score,Class,tile_id,filename,...,height,left,top,pixel_size_x,pixel_size_y,crs,global_x_min,global_y_min,global_x_max,global_y_max
0,29,42,382.697100,312.86465,512.40295,472.19528,0.788418,construction waste,29_42,tile_29_42.jpg,...,1024,266602.172852,9.863026e+06,0.305176,0.305176,EPSG:32737,266718.962738,9.862930e+06,266758.545822,9.862882e+06
1,31,33,60.977410,186.72801,133.88120,232.27876,0.347023,domestic garbage,31_33,tile_31_33.jpg,...,1024,263789.672852,9.862401e+06,0.305176,0.305176,EPSG:32737,263808.281680,9.862344e+06,263830.530151,9.862330e+06
2,31,33,60.005318,187.15474,131.67242,234.79256,0.573664,construction waste,31_33,tile_31_33.jpg,...,1024,263789.672852,9.862401e+06,0.305176,0.305176,EPSG:32737,263807.985021,9.862344e+06,263829.856085,9.862329e+06
3,65,6,278.078250,513.72156,396.46893,606.74680,0.702488,domestic garbage,65_6,tile_65_6.jpg,...,1024,255352.172852,9.851776e+06,0.305176,0.305176,EPSG:32737,255437.035599,9.851619e+06,255473.165567,9.851591e+06
4,56,37,654.497200,624.95850,697.35230,675.28796,0.439645,domestic garbage,56_37,tile_56_37.jpg,...,1024,265039.672852,9.854588e+06,0.305176,0.305176,EPSG:32737,265239.409546,9.854398e+06,265252.487885,9.854382e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
744,51,40,504.013550,614.90590,554.47060,668.43250,0.484410,domestic garbage,51_40,tile_51_40.jpg,...,1024,265977.172852,9.856151e+06,0.305176,0.305176,EPSG:32737,266130.985580,9.855963e+06,266146.383850,9.855947e+06
745,51,40,414.520450,706.50910,476.34143,753.50616,0.420728,domestic garbage,51_40,tile_51_40.jpg,...,1024,265977.172852,9.856151e+06,0.305176,0.305176,EPSG:32737,266103.674454,9.855935e+06,266122.540720,9.855921e+06
746,46,4,224.373570,705.97845,348.96005,827.33466,0.583834,construction waste,46_4,tile_46_4.jpg,...,1024,254727.172852,9.857713e+06,0.305176,0.305176,EPSG:32737,254795.646231,9.857498e+06,254833.667007,9.857461e+06
747,16,2,458.533450,670.63855,491.75305,709.15247,0.728989,domestic garbage,16_2,tile_16_2.jpg,...,1024,254102.172852,9.867088e+06,0.305176,0.305176,EPSG:32737,254242.106155,9.866884e+06,254252.243973,9.866872e+06


### Only domestic

In [ ]:
# Filter only "domestic garbage" detections
domestic_garbage_df = merged_df[merged_df["Class"] == "domestic garbage"]

# Function to create polygons
def create_polygon(row):
    try:
        polygon = Polygon([
            (row["global_x_min"], row["global_y_min"]),
            (row["global_x_max"], row["global_y_min"]),
            (row["global_x_max"], row["global_y_max"]),
            (row["global_x_min"], row["global_y_max"]),
            (row["global_x_min"], row["global_y_min"])  # Closing the polygon
        ])
        return polygon if polygon.is_valid else None
    except Exception as e:
        print(f"Error creating polygon for row {row.name}: {e}")
        return None

# Apply function to create geometry
domestic_garbage_df["geometry"] = domestic_garbage_df.apply(create_polygon, axis=1)

# Drop invalid geometries
domestic_garbage_df = domestic_garbage_df.dropna(subset=["geometry"])

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(domestic_garbage_df, geometry="geometry", crs="EPSG:32737")

# Save as GeoJSON for GIS visualization
gdf.to_file(MMD + "MMD_domestic_garbage.geojson", driver="GeoJSON")

In [44]:
merged_df['Class'].unique()

array(['construction waste', 'domestic garbage', 'agriculture forestry',
       'disposed garbage'], dtype=object)

### All detection class

In [9]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon

# Keep all classes (remove filtering for "domestic garbage")
all_classes_df = merged_df.copy()

# Function to create polygons
def create_polygon(row):
    try:
        polygon = Polygon([
            (row["global_x_min"], row["global_y_min"]),
            (row["global_x_max"], row["global_y_min"]),
            (row["global_x_max"], row["global_y_max"]),
            (row["global_x_min"], row["global_y_max"]),
            (row["global_x_min"], row["global_y_min"])  # Closing the polygon
        ])
        return polygon if polygon.is_valid else None
    except Exception as e:
        print(f"Error creating polygon for row {row.name}: {e}")
        return None

# Apply function to create geometry
all_classes_df["geometry"] = all_classes_df.apply(create_polygon, axis=1)

# Drop invalid geometries
all_classes_df = all_classes_df.dropna(subset=["geometry"])

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(all_classes_df, geometry="geometry", crs="EPSG:32737")

# Save as GeoJSON for GIS visualization
gdf.to_file(MMD + "MMD_all_classes.geojson", driver="GeoJSON")

print("✅ All classes saved successfully as a GeoJSON!")


✅ All classes saved successfully as a GeoJSON!
